# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [ ]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [ ]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(5, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA
    # Los imports y la clase adicional deberían estar en otra celda de código, pero
    # > Es obligatorio completar únicamente las secciones indicadas, sin agregar contenido adicional.

    from itertools import count
    from queue import PriorityQueue
    from typing import Self

    class NodeHanoiPriorityQueue(PriorityQueue[tuple[float, int, NodeHanoi]]):

        def __init__(self, statesHanoi_goal: StatesHanoi) -> None:
            super().__init__()
            self._count: count = count()
            self._statesHanoi_goal: StatesHanoi = statesHanoi_goal

        @staticmethod
        def _priority_cost(statesHanoi_from: StatesHanoi, statesHanoi_to: StatesHanoi) -> float:
            return statesHanoi_to.accumulated_cost - statesHanoi_from.accumulated_cost

        @staticmethod
        def _priority_heuristic(statesHanoi_to: StatesHanoi, statesHanoi_goal: StatesHanoi) -> float:
            heuristic: float = 0
            for rod_to, rod_goal in zip(statesHanoi_to.rods, statesHanoi_goal.rods):
                for disk_to, disk_goal in zip(rod_to, rod_goal):
                    if disk_to == disk_goal:
                        heuristic -= 1
                    else:
                        break
            return heuristic

        def _priority(self, statesHanoi_from: StatesHanoi, statesHanoi_to: StatesHanoi) -> float:
            return (
                NodeHanoiPriorityQueue._priority_cost(statesHanoi_from, statesHanoi_to) +
                NodeHanoiPriorityQueue._priority_heuristic(statesHanoi_to, self._statesHanoi_goal)
            )

        def put(self, nodeHanoi_from: NodeHanoi, nodeHanoi_to: NodeHanoi) -> Self:
            priority: float = self._priority(nodeHanoi_from.state, nodeHanoi_to.state)
            super().put((priority, next(self._count), nodeHanoi_to))
            return self

    # Este es el código actual

    metrics = {
        "cost_total": 0,
        "max_depth": 0,
        "nodes_explored": 0,
        "nodes_in_frontier": 0,
        "solution_found": False,
        "states_visited": 0,
    }

    nodeHanoi_initial: NodeHanoi = NodeHanoi(initial_state)
    nodeHanoiPriorityQueue: NodeHanoiPriorityQueue = NodeHanoiPriorityQueue(goal_state)
    nodeHanoiPriorityQueue.put(nodeHanoi_initial, nodeHanoi_initial)
    statesHanoiSet: set[StatesHanoi] = {nodeHanoi_initial.state}

    while not nodeHanoiPriorityQueue.empty():

        metrics["nodes_explored"] += 1
        _, _, nodeHanoi_from = nodeHanoiPriorityQueue.get()

        statesHanoi_from: StatesHanoi = nodeHanoi_from.state
        statesHanoiSet.add(statesHanoi_from)

        if problem.goal_test(statesHanoi_from):
            metrics["cost_total"] = statesHanoi_from.accumulated_cost
            metrics["max_depth"] = nodeHanoi_from.depth
            metrics["nodes_in_frontier"] = nodeHanoiPriorityQueue.qsize()
            metrics["solution_found"] = True
            metrics["states_visited"] = len(statesHanoiSet)
            return nodeHanoi_from, metrics

        for nodeHanoi_to in nodeHanoi_from.expand(problem):
            if nodeHanoi_to.state not in statesHanoiSet:
                nodeHanoiPriorityQueue.put(nodeHanoi_from, nodeHanoi_to)

    return None, metrics

Se prueba la función:

In [ ]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [ ]:
for key, value in metrics.items():
    print(f"{key}: {value}")

cost_total: 31.0
max_depth: 31
nodes_explored: 300
nodes_in_frontier: 97
solution_found: True
states_visited: 116


Veamos el camino de estados desde el principio a la solución:

In [ ]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [ ]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
